# Physics-Informed Neural Networks (PINNs) for 2D Axisymmetric Taylor Dispersion

### Physical Motivation
Imagine a bolus of solute is injected rapidly into a laminar pressure driven flow inside a cylindrical tube of radius $R$, creating a momentary sharp peak in solute concentration. As this solute is advected downstream, two competing transport mechanisms govern its evolution:
1. Axial Advective Transport: Fluid at the centerline ($r = 0$) moves at twice the cross-sectional mean speed ($u_{\max} = 2 U_{\text{avg}}$), while fluid at the impermeable wall ($r = R$) is stationary ($u = 0$). This parabolic velocity profile stretches the initial solute plug into a long, bullet-shaped filament.
2. Radial Molecular Diffusion: The resulting radial concentration gradients drive solute molecules from the fast-moving core toward the slow-moving wall near the leading edge, and from the wall back into the core near the trailing edge.

At times exceeding the radial diffusion timescale ($t \gtrsim 0.25\,R^2 / D$), radial diffusion homogenizes cross-sectional variations. Remarkably, the cross-sectional area-averaged concentration $\bar{c}(z, t)$ evolves as a symmetric 1D Gaussian pulse translating at the mean flow speed $U_{\text{avg}}$ with an enhanced **effective dispersion coefficient** (Taylor, *Proc. R. Soc. A* 1953; Aris, *Proc. R. Soc. A* 1956):

$$D_{\text{eff}} = D + \frac{U_{\text{avg}}^2 R^2}{48 D}$$

---

### Notebook Overview & Numerical Strategy
In this notebook, we build a mesh-free Physics-Informed Neural Network (PINN) model using Pytorch to solve the full transient 2D axisymmetric advection-diffusion equation from first principles—without relying on external CFD or finite-difference training data.

To capture sharp early-time parabolic stretching and late-time Gaussian dispersion without numerical dissipation or mass loss, we combine four key techniques:
* Co-Moving Reference Frame: Making the change of variables in our system ($\xi = z^* - t^*$) allows us to transform into a "Galilean frame" where in this case, moving at the mean flow velocity eliminates long-range bulk translation which isolates relative shear-induced spreading ($u_{\text{rel}}(r^*) = 1 - 2{r^*}^2$) inside a compact spatial window.
* Axisymmetric Singularity Treatment: Applying L'Hôpital's rule along the centerline ($r^* = 0$) ensures smooth radial diffusion gradients without $1/r^*$ division singularities.
* Causal Curriculum + Mass Conservation: Pairing chronological residual weighting during Adam with a deterministic mass conservation constraint and an L-BFGS second-order polish preserves total solute mass to within < 0.05%.
* Asymptotic Benchmarking: Integrating the predicted 2D field across the tube radius ($\bar{c} = 2\int_0^1 c\,r^*\,dr^*$) verifies direct convergence onto the exact 1D Aris-Taylor analytical limit.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

### 2. Physics-Informed Neural Network Architecture

We parameterize the continuous concentration field $c(r^*, \xi, t^*)$ using a 6-hidden-layer Multilayer Perceptron (MLP) with 100 neurons per layer (`TaylorDispersionNet`). Several targeted architectural choices ensure stable second-order automatic differentiation and physical consistency:

* Domain Input Normalization: Physical inputs $(r^*, \xi, t^*)$ are linearly mapped onto $[-1, 1]^3$ inside the forward pass so the hidden layers operate within their active, non-saturated dynamic range regardless of domain length.
* Smooth `Tanh` Activations & Xavier Initialization: Because the advection-diffusion PDE requires second-order spatial derivatives ($\partial^2 c / \partial \xi^2$ and $\partial^2 c / \partial {r^*}^2$), infinitely differentiable `Tanh` activation functions are preffered. Pairing this with Xavier normal weight initialization help set initial random weights of a neural network so that the variance of the activations and gradients remain roughly the same across different layers. This helps prevent vanishing or exploding gradients during backpropagation.
* Hard-Enforced Non-Negativity (`Softplus`): Applying a smooth `Softplus` output activation ($\beta = 5.0$) guarantees strictly non-negative solute concentrations ($c > 0$) everywhere, suppressing unphysical undershoot ripples near sharp initial concentration fronts.


In [2]:
class TaylorDispersionNet(nn.Module):
    def __init__(self, r_max=1.0, xi_min=-6.0, xi_max=6.0, t_max=10.0):
        super().__init__()
        self.r_max = r_max
        self.xi_min = xi_min
        self.xi_max = xi_max
        self.t_max = t_max

        self.layer1 = nn.Linear(3, 100)
        self.layer2 = nn.Linear(100, 100)
        self.layer3 = nn.Linear(100, 100)
        self.layer4 = nn.Linear(100, 100)
        self.layer5 = nn.Linear(100, 100)
        self.layer6 = nn.Linear(100, 100)
        self.layer7 = nn.Linear(100, 1)
        self.act = nn.Tanh()

        for layer in [self.layer1, self.layer2, self.layer3, self.layer4, self.layer5, self.layer6, self.layer7]:
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, r, xi, t):
        r_norm  = 2.0 * (r / self.r_max) - 1.0
        xi_norm = 2.0 * ((xi - self.xi_min) / (self.xi_max - self.xi_min)) - 1.0
        t_norm  = 2.0 * (t / self.t_max) - 1.0

        x = torch.cat((r_norm, xi_norm, t_norm), dim=1)
        x = self.act(self.layer1(x))
        x = self.act(self.layer2(x))
        x = self.act(self.layer3(x))
        x = self.act(self.layer4(x))
        x = self.act(self.layer5(x))
        x = self.act(self.layer6(x))
        output = self.layer7(x)
        return torch.nn.functional.softplus(output, beta = 5.0)  # Ensure non-negative concentration




### 1. Governing Equations & Co-Moving Frame Transformation

We consider transient axisymmetric solute transport in a cylindrical tube of radius $R$ under steady, fully developed Hagen-Poiseuille flow with cross-sectional mean velocity $U_{\text{avg}}$:

$$\frac{\partial c}{\partial t} + 2 U_{\text{avg}}\left(1 - \frac{r^2}{R^2}\right)\frac{\partial c}{\partial z} = D\left(\frac{\partial^2 c}{\partial z^2} + \frac{\partial^2 c}{\partial r^2} + \frac{1}{r}\frac{\partial c}{\partial r}\right)$$

#### Dimensionless Scales
We nondimensionalize radial position, axial position, and time using the tube radius $R$ and the advective timescale $R / U_{\text{avg}}$:

$$r^* = \frac{r}{R}, \qquad z^* = \frac{z}{R}, \qquad t^* = \frac{U_{\text{avg}}\,t}{R}, \qquad \text{Pe} = \frac{U_{\text{avg}} R}{D}$$

Here, $t^*$ represents both dimensionless advective time and the axial distance (in tube radii) traveled by the bulk flow.

#### Galilean Transformation ($\xi = z^* - t^*$)
To eliminate long-range bulk advection and isolate relative shear-induced spreading, we transform into a reference frame moving at the cross-sectional mean velocity ($\bar{u}^* = 1$):

$$\xi = \frac{z - U_{\text{avg}}\,t}{R} = z^* - t^*$$

Applying the chain rule ($\partial_{t^*} \to \partial_{t^*} - \partial_\xi$ and $\partial_{z^*} \to \partial_\xi$) yields the co-moving dimensionless advection-diffusion equation:

$$\frac{\partial c}{\partial t^*} + (1 - 2{r^*}^2)\frac{\partial c}{\partial \xi} = \frac{1}{\text{Pe}}\left(\frac{\partial^2 c}{\partial \xi^2} + \frac{\partial^2 c}{\partial {r^*}^2} + \frac{1}{r^*}\frac{\partial c}{\partial r^*}\right)$$


In [ ]:
def pde_residual(model, r, xi, t, Pe):
    r.requires_grad = True
    xi.requires_grad = True
    t.requires_grad = True

    c = model(r, xi, t)

    c_t   = torch.autograd.grad(c, t, torch.ones_like(c), create_graph=True)[0]
    c_xi  = torch.autograd.grad(c, xi, torch.ones_like(c), create_graph=True)[0]
    c_r   = torch.autograd.grad(c, r, torch.ones_like(c), create_graph=True)[0]
    c_xixi = torch.autograd.grad(c_xi, xi, torch.ones_like(c_xi), create_graph=True)[0]
    c_rr  = torch.autograd.grad(c_r, r, torch.ones_like(c_r), create_graph=True)[0]

    # Axisymmetric radial diffusion with L'Hôpital's rule at r = 0
    eps = 1e-7
    radial_diffusion = torch.where(r > eps, c_rr + (1.0 / r) * c_r, 2.0 * c_rr) 

    # Relative velocity in moving frame: u_z - u_avg = 2*(1 - r^2) - 1 = 1 - 2*r^2
    u_rel = 1.0 - 2.0 * (r ** 2)
    return c_t + u_rel * c_xi - (1.0 / Pe) * (c_xixi + radial_diffusion)



### 3. Governing PDE Residual & Boundary Conditions

To train the network without external simulation data, we construct automatic differentiation residuals enforcing the governing physics across the interior domain and boundaries:

* Co-Moving Transport Residual (`pde_residual`): Evaluates $\mathcal{R}_{\text{PDE}} = \frac{\partial c}{\partial t^*} + (1 - 2{r^*}^2)\frac{\partial c}{\partial \xi} - \frac{1}{\text{Pe}}\left(\frac{\partial^2 c}{\partial \xi^2} + \frac{\partial^2 c}{\partial {r^*}^2} + \frac{1}{r^*}\frac{\partial c}{\partial r^*}\right)$ via nested `torch.autograd.grad` calls (`create_graph=True`). Along the centerline ($r^* = 0$), L'Hôpital's rule ($\frac{1}{r^*}\frac{\partial c}{\partial r^*} \to \frac{\partial^2 c}{\partial {r^*}^2}$) prevents $1/r^*$ division singularities.
* Smooth Initial Solute Plug (`initial_condition`): Models a finite top-hat injection of concentration $C_0 = 1$ spanning $\xi \in [-0.5, 0.5]$ at $t^* = 0$ using opposing hyperbolic tangent profiles ($\text{sharpness} = 12.0$) to avoid infinite spatial gradients at the step edges.
* Physical Boundary Conditions: Enforces zero radial mass flux ($\frac{\partial c}{\partial r^*} = 0$) at the impermeable pipe wall ($r^* = 1$), radial symmetry ($\frac{\partial c}{\partial r^*} = 0$) at the centerline ($r^* = 0$), and unperturbed far-field conditions ($c = 0$) at the upstream and downstream domain boundaries ($\xi = \xi_{\min}, \xi_{\max}$).


In [4]:
# 1. SHIFTED TOP-HAT INITIAL CONDITION (xi ∈ [-0.5, 0.5])
def initial_condition(r, xi, t, C0=1.0, xi_start=-0.5, xi_end=0.5, sharpness=8.0):
    left_edge  = 0.5 * (1.0 + torch.tanh(sharpness * (xi - xi_start)))
    right_edge = 0.5 * (1.0 - torch.tanh(sharpness * (xi - xi_end)))
    return C0 * left_edge * right_edge


# 2. INLET BOUNDARY CONDITION at xi = xi_min (t > 0)
# Clean fluid enters the domain behind the moving slug.
def inlet_boundary_condition(r, xi, t):
    return torch.zeros_like(xi)  # c(r, xi_min, t) = 0

# 3. OUTLET BOUNDARY CONDITION at xi = xi_max
# Enforcing a zero-gradient boundary condition prevents artificial back-reflection.
def outlet_boundary_condition(r, xi, t):
    return torch.zeros_like(xi) # Target loss drives c_xi -> 0 at xi = xi_max

# 4. IMPERMEABLE WALL CONDITION at r = 1.0 (Dimensionless Pipe Wall)
# Because it is dimensionless, the wall is strictly at r = 1.0.
def wall_boundary_condition(model, r, xi, t):
    r.requires_grad = True
    c = model(r, xi, t)
    c_r = torch.autograd.grad(c, r, torch.ones_like(c), create_graph=True)[0]
    return c_r  # Target loss drives c_r -> 0

# 5. CENTERLINE SYMMETRY CONDITION at r = 0.0
# Smooth profile across the axis.
def symmetry_boundary_condition(model, r, xi, t):
    r.requires_grad = True
    c = model(r, xi, t)
    c_r = torch.autograd.grad(c, r, torch.ones_like(c), create_graph=True)[0]
    return c_r  # Target loss drives c_r -> 0




### 4. Composite Physics-Informed Loss Formulation

Standard unweighted mean-squared error (MSE) losses often fail on transient advection-diffusion problems because the network can minimize late-time residuals by decaying to the trivial zero solution ($c \approx 0$). To guarantee accurate wave propagation and solute conservation, we formulate a composite objective:

* Boundary & Initial Condition MSE Losses: Penalizes deviations from the initial solute profile at $t^* = 0$ and the radial/axial boundary constraints across all $t^* \in [0, T_{\max}]$.
* Causal PDE Residual Weighting (`causal_pde_loss`): Sorts interior collocation points chronologically and weights each temporal slab $i$ by $w_i = \exp\!\left(-\epsilon_{\text{causal}} \sum_{k < i} \mathcal{L}_{\text{PDE}, k}\right)$ (Wang et al., 2024). This forces the optimizer to resolve early-time shear stretching before minimizing late-time residuals.
* Global Solute Mass Conservation (`mass_conservation_loss`): Numerically integrates the cross-sectional area-averaged concentration across the domain at deterministic time slices:
  $$M(t^*) = 2 \int_{\xi_{\min}}^{\xi_{\max}} \int_0^1 c(r^*, \xi, t^*)\,r^*\,dr^*\,d\xi = M_0$$
  Penalizing deviations from the initial injected mass $M_0 = C_0(\xi_{\text{end}} - \xi_{\text{start}})$ prevents artificial numerical dissipation over long integration horizons.


In [5]:
def initial_condition_loss(model, r, xi, t, C0=1.0, xi_start=-0.5, xi_end=0.5):
    c_pred = model(r, xi, t)
    c_init = initial_condition(r, xi, t, C0=C0, xi_start=xi_start, xi_end=xi_end)
    loss_ic = torch.mean((c_pred - c_init) ** 2)
    return loss_ic

def inlet_boundary_condition_loss(model, r, xi, t):
    c_pred = model(r, xi, t)
    c_inlet = inlet_boundary_condition(r, xi, t)
    loss_inlet = torch.mean((c_pred - c_inlet) ** 2)
    return loss_inlet

def outlet_boundary_condition_loss(model, r, xi, t):
    c_pred = model(r, xi, t)
    # Removed 'model' to match def outlet_boundary_condition(r, xi, t)
    c_outlet = outlet_boundary_condition(r, xi, t)
    loss_outlet = torch.mean((c_pred - c_outlet) ** 2)
    return loss_outlet

def wall_boundary_condition_loss(model, r, xi, t):
    c_r_pred = wall_boundary_condition(model, r, xi, t)
    loss_wall = torch.mean(c_r_pred ** 2)
    return loss_wall

def symmetry_boundary_condition_loss(model, r, xi, t):
    c_r_pred = symmetry_boundary_condition(model, r, xi, t)
    loss_symmetry = torch.mean(c_r_pred ** 2)
    return loss_symmetry

# 1. Deterministic Mass Conservation Loss (Prevents flatlining at t* > 2)
def mass_conservation_loss(model, xi_min=-6.0, xi_max=6.0, T_max=6.0, target_mass=1.0, n_t=20, n_r=32, n_xi=128):
    t_sample = torch.linspace(0, T_max, n_t, device=device)
    r_grid   = torch.linspace(0, 1.0, n_r, device=device)
    xi_grid  = torch.linspace(xi_min, xi_max, n_xi, device=device)
    
    T_m, R_m, XI_m = torch.meshgrid(t_sample, r_grid, xi_grid, indexing='ij')
    c_pred = model(R_m.reshape(-1, 1), XI_m.reshape(-1, 1), T_m.reshape(-1, 1)).view(n_t, n_r, n_xi)
    
    c_bar  = 2.0 * torch.trapz(c_pred * R_m, r_grid, dim=1)
    mass_t = torch.trapz(c_bar, xi_grid, dim=1)
    return torch.mean((mass_t - target_mass) ** 2)

# 2. Causal PDE Loss (Forces chronological propagation past t* = 2)
def causal_pde_loss(model, r_pde, xi_pde, t_pde, Pe, num_chunks=16, eps_causal=5.0):
    idx = torch.argsort(t_pde.squeeze())
    r_s, xi_s, t_s = r_pde[idx], xi_pde[idx], t_pde[idx]
    
    res = pde_residual(model, r_s, xi_s, t_s, Pe) ** 2
    chunks = torch.chunk(res, num_chunks)
    chunk_losses = torch.stack([torch.mean(c) for c in chunks])
    
    cum_loss = torch.cumsum(chunk_losses, dim=0) - chunk_losses
    weights = torch.exp(-eps_causal * cum_loss).detach()
    return torch.mean(weights * chunk_losses)






In [6]:
import torch

# ==========================================
# 1. DOMAIN BOUNDS (Co-Moving Frame)
# ==========================================
R_min, R_max = 0.0, 1.0
xi_min, xi_max = -6.0, 6.0
T_min, T_max = 0.0, 10.0

# Helper function for uniform sampling on [low, high]
def sample_uniform(n, low, high):
    return low + (high - low) * torch.rand(n, 1)

# ==========================================
# 2. INTERIOR PDE COLLOCATION POINTS
# ==========================================
N_pde = 30000
r_pde  = sample_uniform(N_pde, R_min, R_max)
xi_pde = sample_uniform(N_pde, xi_min, xi_max)
t_pde  = sample_uniform(N_pde, T_min, T_max)

# ==========================================
# 3. INITIAL CONDITION POINTS (t = 0)
# ==========================================
# Combine global points across [-8, 8] with dense points near the slug [-1.5, 1.5]
N_ic_global = 2000
N_ic_local  = 2000

r_ic  = sample_uniform(N_ic_global + N_ic_local, R_min, R_max)
xi_ic = torch.cat([
    sample_uniform(N_ic_global, xi_min, xi_max),
    sample_uniform(N_ic_local, -1.5, 1.5)
], dim=0)
t_ic  = torch.zeros(N_ic_global + N_ic_local, 1)

# ==========================================
# 4. AXIAL FAR-FIELD BOUNDARIES (xi = -8 and xi = +8)
# ==========================================
N_bc = 2000

# Inlet (xi = -6.0)
r_inlet  = sample_uniform(N_bc, R_min, R_max)
xi_inlet = torch.full((N_bc, 1), xi_min)
t_inlet  = sample_uniform(N_bc, T_min, T_max)

# Outlet (xi = +6.0)
r_outlet  = sample_uniform(N_bc, R_min, R_max)
xi_outlet = torch.full((N_bc, 1), xi_max)
t_outlet  = sample_uniform(N_bc, T_min, T_max)

# ==========================================
# 5. RADIAL BOUNDARIES (r = 1.0 Wall & r = 0.0 Centerline)
# ==========================================
# Pipe Wall (r = 1.0)
r_wall  = torch.full((N_bc, 1), R_max)
xi_wall = sample_uniform(N_bc, xi_min, xi_max)
t_wall  = sample_uniform(N_bc, T_min, T_max)

# Centerline Symmetry (r = 0.0)
r_sym  = torch.full((N_bc, 1), R_min)
xi_sym = sample_uniform(N_bc, xi_min, xi_max)
t_sym  = sample_uniform(N_bc, T_min, T_max)


In [8]:
import torch

# ==========================================
# 1. DEVICE & GLOBAL SIMULATION PARAMETERS (Co-Moving Frame)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Pe = 5.0            # Peclet number (Pe = U_avg * R / D)
R_max = 1.0         # Dimensionless tube radius
xi_min = -6.0       # Co-moving axial lower bound (xi = z* - t*)
xi_max = 6.0        # Co-moving axial upper bound
T_max = 10.0        # Dimensionless advective time (tau = t * U_avg / R)
xi_start = -0.5     # Initial slug left edge
xi_end = 0.5        # Initial slug right edge
C0 = 1.0            # Initial slug concentration
target_mass = C0 * (xi_end - xi_start)

# ==========================================
# 2. INSTANTIATE MODEL & MOVE DATA TO DEVICE
# ==========================================
model = TaylorDispersionNet(r_max=R_max, xi_min=xi_min, xi_max=xi_max, t_max=T_max).to(device)

r_pde, xi_pde, t_pde = r_pde.to(device), xi_pde.to(device), t_pde.to(device)
r_ic, xi_ic, t_ic = r_ic.to(device), xi_ic.to(device), t_ic.to(device)
r_inlet, xi_inlet, t_inlet = r_inlet.to(device), xi_inlet.to(device), t_inlet.to(device)
r_outlet, xi_outlet, t_outlet = r_outlet.to(device), xi_outlet.to(device), t_outlet.to(device)
r_wall, xi_wall, t_wall = r_wall.to(device), xi_wall.to(device), t_wall.to(device)
r_sym, xi_sym, t_sym = r_sym.to(device), xi_sym.to(device), t_sym.to(device)

# ==========================================
# 3. TOTAL LOSS FUNCTION (Causal Adam -> Unweighted L-BFGS)
# ==========================================
def compute_total_loss(use_causal=True):
    # 1. PDE Loss: Causal during Adam, standard unweighted MSE during L-BFGS
    if use_causal:
        loss_pde = causal_pde_loss(model, r_pde, xi_pde, t_pde, Pe, num_chunks=16, eps_causal=5.0)
    else:
        res_pde  = pde_residual(model, r_pde, xi_pde, t_pde, Pe)
        loss_pde = torch.mean(res_pde ** 2)

    # 2. Initial Condition Loss
    loss_ic = initial_condition_loss(model, r_ic, xi_ic, t_ic, C0=C0, xi_start=xi_start, xi_end=xi_end)

    # 3. Boundary Condition Losses
    loss_inlet  = inlet_boundary_condition_loss(model, r_inlet, xi_inlet, t_inlet)
    loss_outlet = outlet_boundary_condition_loss(model, r_outlet, xi_outlet, t_outlet)
    loss_wall   = wall_boundary_condition_loss(model, r_wall, xi_wall, t_wall)
    loss_sym    = symmetry_boundary_condition_loss(model, r_sym, xi_sym, t_sym)

    # 4. Mass Conservation Loss (Prevents flatlining at t* > 2)
    loss_mass = mass_conservation_loss(model, xi_min=xi_min, xi_max=xi_max, T_max=T_max, target_mass=target_mass, n_t=20, n_r=32, n_xi=128)

    # Add ALL loss terms into total_loss
    total_loss = (
        1.0 * loss_pde +
        10.0 * loss_ic +
        10.0 * loss_mass +
        1.0 * (loss_inlet + loss_outlet + loss_wall + loss_sym)
    )
    return total_loss, loss_pde, loss_ic, loss_mass

import torch

# ==========================================
# 1. DEVICE & GLOBAL SIMULATION PARAMETERS (Co-Moving Frame)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Pe = 5.0            # Peclet number (Pe = U_avg * R / D)
R_max = 1.0         # Dimensionless tube radius
xi_min = -6.0       # Co-moving axial lower bound (xi = z* - t*)
xi_max = 6.0        # Co-moving axial upper bound
T_max = 10.0        # Dimensionless advective time (tau = t * U_avg / R)
xi_start = -0.5     # Initial slug left edge
xi_end = 0.5        # Initial slug right edge
C0 = 1.0            # Initial slug concentration
target_mass = C0 * (xi_end - xi_start)

# ==========================================
# 2. INSTANTIATE MODEL & MOVE DATA TO DEVICE
# ==========================================
model = TaylorDispersionNet(r_max=R_max, xi_min=xi_min, xi_max=xi_max, t_max=T_max).to(device)

r_pde, xi_pde, t_pde = r_pde.to(device), xi_pde.to(device), t_pde.to(device)
r_ic, xi_ic, t_ic = r_ic.to(device), xi_ic.to(device), t_ic.to(device)
r_inlet, xi_inlet, t_inlet = r_inlet.to(device), xi_inlet.to(device), t_inlet.to(device)
r_outlet, xi_outlet, t_outlet = r_outlet.to(device), xi_outlet.to(device), t_outlet.to(device)
r_wall, xi_wall, t_wall = r_wall.to(device), xi_wall.to(device), t_wall.to(device)
r_sym, xi_sym, t_sym = r_sym.to(device), xi_sym.to(device), t_sym.to(device)

# ==========================================
# 3. TOTAL LOSS FUNCTION (Causal Adam -> Unweighted L-BFGS)
# ==========================================
def compute_total_loss(use_causal=True):
    # 1. PDE Loss: Causal during Phase 1A, standard unweighted MSE during Phase 1B & 2
    if use_causal:
        loss_pde = causal_pde_loss(model, r_pde, xi_pde, t_pde, Pe, num_chunks=16, eps_causal=5.0)
    else:
        res_pde  = pde_residual(model, r_pde, xi_pde, t_pde, Pe)
        loss_pde = torch.mean(res_pde ** 2)

    # 2. Initial Condition Loss
    loss_ic = initial_condition_loss(model, r_ic, xi_ic, t_ic, C0=C0, xi_start=xi_start, xi_end=xi_end)

    # 3. Boundary Condition Losses
    loss_inlet  = inlet_boundary_condition_loss(model, r_inlet, xi_inlet, t_inlet)
    loss_outlet = outlet_boundary_condition_loss(model, r_outlet, xi_outlet, t_outlet)
    loss_wall   = wall_boundary_condition_loss(model, r_wall, xi_wall, t_wall)
    loss_sym    = symmetry_boundary_condition_loss(model, r_sym, xi_sym, t_sym)

    # 4. Mass Conservation Loss (n_xi=256 prevents trapezoidal grid aliasing)
    loss_mass = mass_conservation_loss(
        model, xi_min=xi_min, xi_max=xi_max, T_max=T_max,
        target_mass=target_mass, n_t=20, n_r=32, n_xi=256
    )

    total_loss = (
        1.0 * loss_pde +
        10.0 * loss_ic +
        10.0 * loss_mass +
        1.0 * (loss_inlet + loss_outlet + loss_wall + loss_sym)
    )
    return total_loss, loss_pde, loss_ic, loss_mass

# ==========================================
# 4A. PHASE 1A: CAUSAL ADAM (Global Wave Propagation)
# ==========================================
optimizer_adam = torch.optim.Adam(model.parameters(), lr=0.001)
adam_epochs = 2500

print("--- Starting Phase 1A: Causal Adam ---")
for epoch in range(adam_epochs):
    optimizer_adam.zero_grad()
    
    total_loss, loss_pde, loss_ic, loss_mass = compute_total_loss(use_causal=True)
    
    total_loss.backward()
    optimizer_adam.step()
    
    if (epoch + 1) % 500 == 0: 
        print(
            f"Causal Adam [{epoch+1}/{adam_epochs}] | "
            f"Total: {total_loss.item():.6f} | "
            f"PDE: {loss_pde.item():.6f} | "
            f"IC: {loss_ic.item():.6f} | "
            f"Mass: {loss_mass.item():.6f}"
        )
        # Resample interior collocation points to prevent overfitting
        r_pde  = sample_uniform(N_pde, 0.0, R_max).to(device)
        xi_pde = sample_uniform(N_pde, xi_min, xi_max).to(device)
        t_pde  = sample_uniform(N_pde, 0.0, T_max).to(device)

# ==========================================
# 4B. PHASE 1B: UNWEIGHTED ADAM WARM-UP (Smooth Handoff to L-BFGS)
# ==========================================
print("\n--- Starting Phase 1B: Unweighted Adam Warm-Up ---")
for epoch in range(500):
    optimizer_adam.zero_grad()
    
    total_loss, loss_pde, loss_ic, loss_mass = compute_total_loss(use_causal=False)
    
    total_loss.backward()
    optimizer_adam.step()

print(
    f"Warm-Up Complete | "
    f"Total: {total_loss.item():.6f} | "
    f"PDE: {loss_pde.item():.6f} | "
    f"IC: {loss_ic.item():.6f} | "
    f"Mass: {loss_mass.item():.6f}"
)

# ==========================================
# 5. PHASE 2: L-BFGS OPTIMIZER (High-Precision Fine Tuning)
# ==========================================
optimizer_lbfgs = torch.optim.LBFGS(
    model.parameters(), 
    lr=1.0, 
    max_iter=3500, 
    max_eval=4000, 
    history_size=100,
    tolerance_grad=1e-12, 
    tolerance_change=1e-12,
    line_search_fn="strong_wolfe"
)

lbfgs_step = 0
def closure():
    global lbfgs_step
    optimizer_lbfgs.zero_grad()
    
    total_loss, loss_pde, loss_ic, loss_mass = compute_total_loss(use_causal=False)
    total_loss.backward()
    
    lbfgs_step += 1
    if lbfgs_step % 250 == 0:
        print(
            f"L-BFGS Step [{lbfgs_step}] | "
            f"Total: {total_loss.item():.6f} | "
            f"PDE: {loss_pde.item():.6f} | "
            f"IC: {loss_ic.item():.6f} | "
            f"Mass: {loss_mass.item():.6f}"
        )
        
    return total_loss

print("\n--- Starting Phase 2: L-BFGS Optimization ---")
optimizer_lbfgs.step(closure)
print(f"Training Complete at L-BFGS Step {lbfgs_step}!")




--- Starting Phase 1A: Causal Adam ---
Causal Adam [500/2500] | Total: 0.125117 | PDE: 0.021049 | IC: 0.006263 | Mass: 0.003290
Causal Adam [1000/2500] | Total: 0.047129 | PDE: 0.024744 | IC: 0.002110 | Mass: 0.000030
Causal Adam [1500/2500] | Total: 0.041462 | PDE: 0.023973 | IC: 0.001685 | Mass: 0.000015
Causal Adam [2000/2500] | Total: 0.044014 | PDE: 0.024977 | IC: 0.001665 | Mass: 0.000216
Causal Adam [2500/2500] | Total: 0.040321 | PDE: 0.024276 | IC: 0.001508 | Mass: 0.000073

--- Starting Phase 1B: Unweighted Adam Warm-Up ---
Warm-Up Complete | Total: 1.466001 | PDE: 0.013476 | IC: 0.140919 | Mass: 0.000816

--- Starting Phase 2: L-BFGS Optimization ---
L-BFGS Step [250] | Total: 0.027718 | PDE: 0.015065 | IC: 0.001219 | Mass: 0.000030
L-BFGS Step [500] | Total: 0.011756 | PDE: 0.008481 | IC: 0.000230 | Mass: 0.000014
L-BFGS Step [750] | Total: 0.008528 | PDE: 0.006304 | IC: 0.000141 | Mass: 0.000010
L-BFGS Step [1000] | Total: 0.006997 | PDE: 0.004844 | IC: 0.000133 | Mass: 0.

In [ ]:
import torch
import numpy as np
from scipy.special import erf
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import PowerNorm
from matplotlib.gridspec import GridSpec

# ==========================================
# 1. SIMULATION PARAMETERS (Match Co-Moving PINN)
# ==========================================
Pe = 5.0           # Peclet number (Pe = U_avg * R / D) — update if testing Pe = 10.0
C0 = 1.0            # Initial slug concentration
xi_start = -0.5     # Initial slug left edge
xi_end = 0.5        # Initial slug right edge
xi_min = -6.0       # Co-moving axial lower bound
xi_max = 6.0        # Co-moving axial upper bound
T_max = 10.0        # Total simulation time (advective units)

# Effective Aris-Taylor dispersion in Option B scaling
D_eff = (1.0 / Pe) + (Pe / 48.0)
initial_mass = C0 * (xi_end - xi_start)

# ==========================================
# 2. GRAETZ-INSPIRED DARK COLOR PALETTE
# ==========================================
BG_DARK = '#0E0E10'        # Matte charcoal figure background
PANEL_BG = '#141417'       # Slightly elevated neutral dark panel
TEXT_COLOR = '#F3F4F6'     # Crisp off-white primary text
MUTED_COLOR = '#9CA3AF'    # Muted gray secondary text
SPINE_COLOR = '#374151'    # Subtle dark slate borders
BLUE_ACCENT = '#38BDF8'    # Sky blue (PINN curve)
PINK_ACCENT = '#EC4899'    # Vibrant magenta-pink (Analytical curve)
AMBER_ACCENT = '#F59E0B'   # Warm amber (Pre-asymptotic badge)
CMAP_THERMAL = 'inferno'   # Matches Graetz forward contour

# ==========================================
# 3. EVALUATE PINN ON CO-MOVING GRID
# ==========================================
model.eval()
device = next(model.parameters()).device

nr = 160   # Radial resolution
nxi = 420  # Co-moving axial resolution
nt = 120   # Total animation frames

r_vals  = np.linspace(0.0, 1.0, nr)
xi_vals = np.linspace(xi_min, xi_max, nxi)
t_vals  = np.linspace(0.0, T_max, nt)

XI_grid, R_grid = np.meshgrid(xi_vals, r_vals)

# Mirror coordinates across centerline (r = 0) for full-pipe visualization
R_full  = np.concatenate([-np.flipud(R_grid[1:]), R_grid], axis=0)
XI_full = np.concatenate([np.flipud(XI_grid[1:]), XI_grid], axis=0)

predictions_2d = []
area_avg_pinn = []
area_avg_analytical = []
total_mass_pinn = []

with torch.no_grad():
    r_flat  = torch.tensor(R_grid.flatten(), dtype=torch.float32, device=device).view(-1, 1)
    xi_flat = torch.tensor(XI_grid.flatten(), dtype=torch.float32, device=device).view(-1, 1)
    
    for t_step in t_vals:
        t_flat = torch.full_like(r_flat, t_step)
        c_pred = model(r_flat, xi_flat, t_flat).cpu().numpy().reshape(nr, nxi)

        #clipping predictions so that numerical artifacts do not produce negative concentrations
        c_pred = np.clip(c_pred, 0, C0)

        # Mirror across centerline for full-tube contour
        c_full = np.concatenate([np.flipud(c_pred[1:]), c_pred], axis=0)
        predictions_2d.append(c_full)
        
        # 1D Cross-Sectional Area Average: 2 * integral_0^1 c(r, xi, t) * r dr
        integrand = c_pred * r_vals[:, None]
        c_bar = 2.0 * np.trapz(integrand, r_vals, axis=0)
        area_avg_pinn.append(c_bar)
        
        # Solute mass conservation check
        mass = np.trapz(c_bar, xi_vals)
        total_mass_pinn.append(mass)
        
        # Exact Analytical Aris-Taylor 1D Solution in Co-Moving Frame (u_rel_avg = 0)
        if t_step == 0:
            c_ana = np.where((xi_vals >= xi_start) & (xi_vals <= xi_end), C0, 0.0)
        else:
            denom = np.sqrt(4.0 * D_eff * t_step)
            term1 = erf((xi_vals - xi_start) / denom)
            term2 = erf((xi_vals - xi_end) / denom)
            c_ana = 0.5 * C0 * (term1 - term2)
            
        area_avg_analytical.append(c_ana)

# ==========================================
# 4. JOURNAL-STYLE ALIGNED GRIDSPEC LAYOUT
# ==========================================
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'mathtext.fontset': 'dejavusans',
    'axes.edgecolor': SPINE_COLOR,
    'axes.linewidth': 0.9,
    'xtick.color': MUTED_COLOR,
    'ytick.color': MUTED_COLOR,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
})

fig = plt.figure(figsize=(10.8, 7.0), dpi=200, facecolor=BG_DARK)

# Dedicated 2x2 GridSpec: Column 0 holds both plots at identical width; Column 1 holds colorbar
gs = GridSpec(
    2, 2, figure=fig,
    width_ratios=[1.0, 0.022],
    height_ratios=[1.05, 1.25],
    wspace=0.03,
    hspace=0.38,
    left=0.08,
    right=0.91,
    top=0.83,
    bottom=0.13
)

ax1 = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])  # Dedicated colorbar axis so ax1 width == ax2 width
ax2 = fig.add_subplot(gs[1, 0])

# Master Header & Subtitle
fig.suptitle(
    'Physics-Informed Neural Networks — Axisymmetric Taylor Dispersion',
    color=TEXT_COLOR, fontsize=14.5, fontweight='bold', y=0.955
)
fig.text(
    0.495, 0.905,
    f'co-moving reference frame ($\\xi = z^* - t^*$)  ·  Pe = {Pe:.0f}  ·  Adam + L-BFGS optimization',
    ha='center', color=MUTED_COLOR, fontsize=9.8
)

# Footer caption
fig.text(
    0.495, 0.025,
    'PyTorch PINN  ·  2D axisymmetric advection-diffusion  ·  Aris-Taylor asymptotic limit',
    ha='center', color=MUTED_COLOR, fontsize=8.8
)

# Nonlinear gamma norm keeps diluted dye clouds (0.05 - 0.25) bright against dark background
norm = PowerNorm(gamma=0.65, vmin=0.0, vmax=C0)
levels = np.linspace(0.0, C0, 80)

# Initial colorbar rendered into dedicated cax
contour0 = ax1.contourf(XI_full, R_full, predictions_2d[0], levels=levels, cmap=CMAP_THERMAL, norm=norm, extend='max')
cbar = fig.colorbar(contour0, cax=cax)
cbar.set_label('Concentration ($C / C_0$)', color=TEXT_COLOR, fontsize=10, labelpad=8)
cbar.outline.set_edgecolor(SPINE_COLOR)
cbar.outline.set_linewidth(0.8)
cbar.ax.yaxis.set_tick_params(color=MUTED_COLOR, labelcolor=MUTED_COLOR, labelsize=9)
cbar.set_ticks([0.0, 0.1, 0.25, 0.5, 0.75, 1.0])

# ==========================================
# 5. ANIMATION UPDATE LOOP
# ==========================================
def update(frame):
    t_curr = t_vals[frame]
    mass_err = abs(total_mass_pinn[frame] - initial_mass) / initial_mass * 100.0
    
    # --- TOP PANEL: 2D Co-Moving Pipe Contour ---
    ax1.clear()
    ax1.set_facecolor(PANEL_BG)
    ax1.contourf(XI_full, R_full, predictions_2d[frame], levels=levels, cmap=CMAP_THERMAL, norm=norm, extend='max')
    
    # Low-threshold iso-concentration contours to track late-time Gaussian spreading
    ax1.contour(
        XI_full, R_full, predictions_2d[frame],
        levels=[0.05, 0.15, 0.35, 0.7], colors='#FFFFFF', linewidths=0.45, alpha=0.35
    )
    
    # Centerline & Pipe Walls
    ax1.axhline(0, color='#FFFFFF', linestyle=':', linewidth=0.9, alpha=0.35)
    ax1.axvline(0, color='#FFFFFF', linestyle=':', linewidth=0.9, alpha=0.25)
    ax1.axhline(1.0, color=SPINE_COLOR, linewidth=1.2)
    ax1.axhline(-1.0, color=SPINE_COLOR, linewidth=1.2)
    
    ax1.set_ylabel('r / R', color=TEXT_COLOR, fontsize=10.5)
    ax1.set_xlim(xi_min, xi_max)
    ax1.set_ylim(-1.0, 1.0)
    ax1.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    ax1.tick_params(labelsize=9.5)
    
    ax1.set_title(
        f'Forward solution: parabolic shear relaxes via radial diffusion   |   $t^* = {t_curr:.2f}$   |   Mass error: {mass_err:.2f}%',
        color=TEXT_COLOR, fontsize=10.8, pad=8
    )
    
    # --- BOTTOM PANEL: 1D Area-Average vs. Analytical Aris-Taylor ---
    ax2.clear()
    ax2.set_facecolor(PANEL_BG)
    
    c_pinn = area_avg_pinn[frame]
    c_ana  = area_avg_analytical[frame]
    
    # Sky-blue translucent fill + primary PINN curve
    ax2.fill_between(xi_vals, 0, c_pinn, color=BLUE_ACCENT, alpha=0.14)
    ax2.plot(xi_vals, c_pinn, color=BLUE_ACCENT, lw=2.3, label='PINN cross-sectional average $\\bar{c}(\\xi, t)$')
    
    # Magenta-pink dashed analytical curve
    ax2.plot(
        xi_vals, c_ana, color=PINK_ACCENT, lw=2.1, linestyle='--',
        label='Aris-Taylor analytical ($D_{\\mathrm{eff}} = \\mathrm{Pe}^{-1} + \\mathrm{Pe}/48$)'
    )
    
    # Regime badge (Amber for pre-asymptotic, Sky-blue for Taylor regime)
    is_pre = t_curr < (0.25 * Pe)
    regime_text = "Pre-asymptotic (advection-sheared)" if is_pre else "Taylor regime (Gaussian dispersion)"
    badge_color = AMBER_ACCENT if is_pre else BLUE_ACCENT
    ax2.text(
        0.985, 0.91, regime_text, transform=ax2.transAxes, ha='right', va='top',
        color=badge_color, fontsize=8.8, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.35', facecolor='#1C1C21', edgecolor=badge_color, linewidth=0.9)
    )
    
    ax2.set_xlabel('Co-moving axial coordinate  $\\xi = z^* - t^*$', color=TEXT_COLOR, fontsize=10.5)
    ax2.set_ylabel('Area-avg conc. ($\\bar{c}$)', color=TEXT_COLOR, fontsize=10.5)
    ax2.set_xlim(xi_min, xi_max)
    ax2.set_ylim(-0.04, C0 * 1.12)
    ax2.tick_params(labelsize=9.5)
    
    # Subtle grid & Graetz-style dark legend box
    ax2.grid(True, linestyle='--', linewidth=0.6, color=SPINE_COLOR, alpha=0.45)
    legend = ax2.legend(
        loc='upper left', frameon=True, facecolor='#1C1C21', edgecolor=SPINE_COLOR, fontsize=9
    )
    for text in legend.get_texts():
        text.set_color(TEXT_COLOR)

# ==========================================
# 6. EXPORT HIGH-RES MP4 FOR LINKEDIN
# ==========================================
ani = animation.FuncAnimation(fig, update, frames=len(predictions_2d), interval=45, blit=False)

try:
    ani.save('taylor_dispersion_Pe5_slow.mp4', writer='ffmpeg', fps=24, dpi=200)
    print("Saved taylor_dispersion_Pe5_slow.mp4 successfully!")
except Exception as e:
    print(f"FFmpeg not found ({e}), falling back to GIF...")
    ani.save('taylor_dispersion_Pe5_slow.gif', writer='pillow', fps=15, dpi=200)
    print("Saved taylor_dispersion_Pe5_slow.gif successfully!")

plt.close()





MovieWriter ffmpeg unavailable; using Pillow instead.


FFmpeg not found (unknown file extension: .mp5), falling back to GIF...
Saved taylor_dispersion_fv4.gif successfully!
